# Density method - $\mu$ interpolation

## 1) Geometry

In [ ]:
###############################################################################
## CODE CELL 1 : Import transformer mesh
###############################################################################

from utils.geometry import transformer
from ngsolve.webgui import Draw

mesh = transformer(maxh = 1e-2)                   #  mesh the transformer geometry
print(f"Region names : {mesh.GetMaterials()}")   # display the regions (materials) labels
print(f"Line names : {mesh.GetBoundaries()}")    # display the lines (boundaries) labels
Draw(mesh)

## 2) Magnetostatic problem

In [ ]:
###############################################################################
## CODE CELL 2 : Define interpolation of the magnetic reluctivity
###############################################################################

from numpy import pi
mu0 = 4e-7 * pi   # magnetic permeability of air (H/m)
mur = 1000        # relative magnetic permeability of the iron    
p = 1

def nu(rho, p=1):
    """ reluctivity depending on the density """
    return 1/mu0 + rho**p * (1/(mu0*mur) - 1/mu0)  # power law interpolation of nu
    # return 1/ (mu0 + rho**p * (mur*mu0 -mu0))    # power law interpolation of mu

# Plot the interpolation
import matplotlib.pyplot as plt
import numpy as np

rho = np.linspace(0,1,100)
plt.plot(rho, nu(rho,p)); plt.xlabel("$\\rho$"); 
plt.ylabel("$\\nu(\\rho)$"); plt.grid(); 
plt.title("Interpolation of the magnetic reluctivity")
plt.show()

In [ ]:
###############################################################################
## CODE CELL 3 : Define the magnetostatic solver
###############################################################################

j = 1e6 # current density in the primary coil (A/m²)

from ngsolve import CF, grad

def curl(v):
    R = CF(((0,1),(-1,0)), dims = (2,2))  # Rotation matrix of angle -pi/2
    return R * grad(v)

from ngsolve import H1, BilinearForm, LinearForm, dx
from utils.solver import solve

def state(rho):
    """ Solve the state equation for a given density field rho """
    fes = H1(mesh, order = 1, dirichlet = "dOmega")
    a, v = fes.TnT()    # define the trial and test functions 
    bf = BilinearForm(curl(v) * (nu(rho) * curl(a)) * dx)  
    lf = LinearForm(j * v * dx("Pp") - j * v * dx("Pm"))
    return solve(bf, lf)

# Test your function
a_rho, Kinv = state(0)
Draw(a_rho)

## 3) Objective function & adjoint problem
### a) Objective function

In [ ]:
from utils.solver import flux

# we want to maximize the secondary flux = minimize its opposite
def f(sol):
    return - flux(sol)

print(f" {f(a_rho) = :.5e} Wb/m")

### b) Adjoint problem

In [ ]:
from ngsolve import dx

def df(a, rho, aStar):
    """ Directional derivative of the objective function in the direction aStar """
    return aStar*dx("Sm")  - aStar*dx("Sp")

from utils.optimization import solve_adjoint

p_rho = solve_adjoint(a_rho, nu,  Kinv, df )
Draw(p_rho)

## 4) Material interpolation

Even if BESO is a ON-OFF method without intermediate materials, the computation of the derivative still depends on the derivative of the interpolation at $\rho = 0$ and $\rho = 1$.

## 5) Gradient of objective function w.r.t $\rho$

In [ ]:
###############################################################################
## CODE CELL 6 : Directional derivative
############################################################################### 

from ngsolve import LinearForm
from utils.solver import curl

def f_prime(state, adjoint, rho):
    """ directional derivative of the objective function """
    fes = rho.space
    drho = fes.TestFunction()
    f_prime = LinearForm(fes)
    f_prime += curl(adjoint) * ( nu(rho).Diff(rho) * curl(state) ) * drho * dx  # directional derivative
    f_prime.Assemble()
    return f_prime  # return the linear form

# Test your function
from ngsolve import L2, GridFunction
rho = GridFunction(L2(mesh, definedon = "Omega_c"))
F_prime = GridFunction(rho.space)
F_prime.vec.data = f_prime(a_rho, p_rho, rho).vec
Draw(F_prime)

In [ ]:
###############################################################################
## CODE CELL 7 : Compute the derivative
############################################################################### 

def f_derivative(a_rho, rho, Kinv):
    """ Encapsulate derivative computation """
    v_rho = solve_adjoint(a_rho, rho, Kinv, df)
    riesz_representer = GridFunction(rho.space)
    riesz_representer.vec.data = f_prime(a_rho, v_rho, rho).vec
    return riesz_representer

# Test the function
Draw(f_derivative(a_rho, rho, Kinv))

## 6) Volume

In [ ]:
###############################################################################
## CODE CELL 10 : Unconstrained optimization algorithm
############################################################################### 

from ngsolve import Integrate
def m(rho, zone = "Omega_c"):
    """ mass function """
    return Integrate(rho*dx(zone), mesh)/Integrate(1, mesh, definedon = mesh.Materials(zone))

def m_prime(rho, zone = "Omega_c"):
    """ directional derivative of the mass function """
    fes = rho.space
    drho = fes.TestFunction()
    mprime = LinearForm(fes)
    mprime += drho * dx(zone)  # directional derivative
    mprime.Assemble()
    return mprime  # return the linear form

def m_derivative(rho):
    """ Encapsulate mass derivative computation """
    riesz_representer = GridFunction(rho.space)
    riesz_representer.vec.data = m_prime(rho).vec
    return riesz_representer

# Test the function
Draw(m_derivative(rho))  # we see the area of each triangle!

## 7) Optimization - BESO update

### Reference
> Querin, O.M., Young, V., Steven, G.P., Xie, Y.M. (2000).  
> *Computational efficiency and validation of bi-directional evolutionary structural optimisation.*  
> Comput. Methods Appl. Mech. Engrg. 189, 559–573



The algorithm follows the steps of Fig. 1 oin the paper:

1. Solve the finite element problem → state $u$
2. Compute the sensitivity criterion per element
3. **Remove** elements with sensitivity < low threshold (`RR × max`)
4. **Add** elements with sensitivity > high threshold (`IR × max`)
5. Check the volume constraint
6. Repeat until convergence



### Parameters (Eq. 4 and 5 of the paper)

The **rejection ratio** $RR$ and the **inclusion ratio** $IR$ evolve as:
$$RR = r_0 + r_1 \cdot SS + a_{RR} \cdot ON \quad (r_0=0,\ r_1=0.001,\ a_{RR}=0.01)$$
$$IR = i_0 - i_1 \cdot SS - a_{IR} \cdot ON \quad (i_0=1,\ i_1=0.01,\ a_{IR}=0.1)$$

In our implementation, these ratios are replaced by:
- `add_precision` and `remove_precision`: percentage of elements to modify per iteration
- `surf_ratio`: target volume constraint
- `init_factor`: correction factor (analogue of $ON$ to avoid oscillations)

> *"An oscillatory state occurs when an element is added in an iteration and the same element  
> is removed in the subsequent iteration"* — Querin et al., Section 2, rule (11)  


In [ ]:
## 7) Optimization - BESO update
from copy import copy
import numpy as np

In [ ]:
def compute_sensitivity(rho):
    u, Kinv = state(rho)
  # 1) objectif function  
    flux_global = f(u)

  # 2) Compute derivative with adjoint method
    grad_elem = f_derivative(u, rho, Kinv).vec.FV().NumPy()[:]
    
    
    #delta_B_elements = sens_Sp - sens_Sm
    delta_B = np.concatenate([[flux_global], grad_elem])

    return delta_B, flux_global


In [ ]:
rho = GridFunction(L2(mesh, definedon = "Omega_c"))

In [ ]:
# ── Initialisation 
V0 = Integrate(1, mesh, definedon = mesh.Materials("Omega_c"))
eVol = Integrate(1, mesh, definedon = mesh.Materials("Omega_c"), element_wise=True).NumPy()[:] / V0
mask = GridFunction(L2(mesh, definedon = mesh.Materials("Omega_c")))
mask.Set(1)
maskOmega_c = mask.vec.FV().NumPy()[:] >= 0.9
N_triangles = np.sum(maskOmega_c)
eVol = eVol[maskOmega_c]

In [ ]:
# ── Sensitivity 

# initialation FEA calcul
delta_B, obj = compute_sensitivity(rho)
initial_energy = delta_B[0]
max_surf = 1

# BESO Parameter
surf_ratio       = 0.1    # = mass_target
surf_precision   = 0.999
add_precision    = 0.1
remove_precision = 0.1
init_factor      = 0.8
N_max            = 100

modified     = list(range(N_triangles))
current_surf = 0.0
loop         = 0

xval         = np.zeros(N_triangles)               # 0 = air, 1 = iron


MemObjective    = []
MemConstraint   = []

scene = Draw(rho, mesh, settings={"Objects": {"Wireframe": False}})


# ── Boucle BESO 
while (current_surf < max_surf * surf_ratio * surf_precision or
       current_surf > max_surf * surf_ratio) and loop < N_max:

    MemObjective.append(obj)
    MemConstraint.append(current_surf / max_surf)

    loop += 1
    ratio = abs(current_surf / max_surf / surf_ratio - 1.0)

    # Ajustement init_factor et surf_precision
    if current_surf > max_surf * surf_ratio:
        init_factor    = 1.1
        surf_precision -= 0.1
    if current_surf < max_surf * surf_ratio * surf_precision and init_factor == 1.1:
        init_factor    = 0.9
        surf_precision -= 0.1

    # Number of elements to be modified
    add_nbr    = round(add_precision    * ratio * N_triangles)
    remove_nbr = round(remove_precision * init_factor * ratio * N_triangles)

    if add_nbr == remove_nbr:
        if current_surf > max_surf * surf_ratio:
            remove_nbr += 1
        else:
            add_nbr += 1
  # Sensitivity by element
    sens = delta_B[1:]
    sens = -sens[maskOmega_c]
# ── Vectorized # use of np.argsort

    modified_mask = np.zeros(N_triangles, dtype=bool)
    modified_mask[modified] = True

    # Elements to add
    candidates_add    = np.where( modified_mask)[0]
    candidates_remove = np.where(~modified_mask)[0]

    if add_nbr > 0 and len(candidates_add) > 0:
        top_add = candidates_add[np.argsort(sens[candidates_add])[-add_nbr:]]
        xval[top_add] = 1.0
        modified_mask[top_add] = False  # retirer de modified

    # Elements to remove
    if remove_nbr > 0 and len(candidates_remove) > 0:
        top_remove = candidates_remove[np.argsort(sens[candidates_remove])[:remove_nbr]]
        xval[top_remove] = 0.0
        modified_mask[top_remove] = True   # remettre dans modified

    # Resynchroniser modified
    modified = list(np.where(modified_mask)[0])

    # Update density
    rho.vec.data.FV().NumPy()[maskOmega_c] = xval

    # FEA 
    delta_B, obj = compute_sensitivity(rho)
    current_energy = delta_B[0]
    if initial_energy == 0.0:
        initial_energy = delta_B[0]

    current_surf = np.sum(eVol * xval)
    scene.Redraw(rho, settings={"Objects": {"Wireframe": False}})

    
# Save la frame
    print(f" It.: {loop:4d}"
          f" | Change: {add_nbr - remove_nbr:+d} ({add_nbr} ajouts, {remove_nbr} suppressions)"
          f" | Objectif: {current_energy / initial_energy:.4f}"
          f" | Surface: {current_surf / max_surf * 100:.2f}%")

## 8) Analysis of the results
u, _ = state(rho)
print(f"\nRésultats BESO : flux = {flux(u):.5e} | vol = {m(rho)*100:.3f}% | itérations = {loop}")

### Evolution objectif et contrainte
fig, ax1 = plt.subplots()
color = 'tab:blue'
ax1.plot(MemObjective, color=color)
ax1.set_xlabel('Itération')
ax1.set_ylabel("Objectif (flux)", color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.plot([s * 100 for s in MemConstraint], color=color)
ax2.set_ylabel("Surface ratio (%)", color=color)
ax2.tick_params(axis='y', labelcolor=color)
fig.tight_layout()
plt.show()

